In [ ]:
import os
import gc
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import timm
from sklearn.preprocessing import LabelEncoder
from scipy.ndimage import gaussian_filter1d

device = torch.device('cpu')
base_path = '/kaggle/input/competitions/birdclef-2026'

model_weights_effnet = '/kaggle/input/datasets/adrianosemerano/birdclef/sed_effnetb2_PhaseB.pth' 
model_weights_convnext = '/kaggle/input/datasets/adrianosemerano/birdclef/sed_convnext_PhaseB.pth'

WEIGHT_EFFNET = 0.50
WEIGHT_CONVNEXT = 0.50

# Reconstruct LabelEncoder
train_csv = os.path.join(base_path, 'train.csv')
df = pd.read_csv(train_csv)
label_encoder = LabelEncoder()
label_encoder.fit(df['primary_label'])
num_classes = len(label_encoder.classes_)

class AudioToSpectrogramGPU(nn.Module):
    def __init__(self, sr=32000, n_mels=256, n_fft=2048, hop_length=512, f_min=40, f_max=15000):
        super(AudioToSpectrogramGPU, self).__init__()
        self.mel_spec = torchaudio.transforms.MelSpectrogram(
            sample_rate=sr, n_fft=n_fft, hop_length=hop_length, f_min=f_min, f_max=f_max, n_mels=n_mels, power=2.0 
        )
        self.eps, self.s, self.alpha, self.delta, self.r = 1e-6, 0.025, 0.98, 2.0, 0.5

    def forward(self, waveform):
        x = self.mel_spec(waveform)
        ema = x.clone()
        for t in range(1, x.size(-1)):
            ema[..., t] = (1 - self.s) * ema[..., t - 1] + self.s * x[..., t]
        x = (x / (self.eps + ema)**self.alpha + self.delta)**self.r - self.delta**self.r
        return (x - x.mean()) / (x.std() + 1e-6)

class AttentivePooling(nn.Module):
    def __init__(self, in_channels, num_classes):
        super(AttentivePooling, self).__init__()
        self.attention = nn.Conv1d(in_channels, num_classes, kernel_size=1, bias=True)
        self.classifier = nn.Conv1d(in_channels, num_classes, kernel_size=1, bias=True)
        
    def forward(self, x):
        att_weights = torch.softmax(self.attention(x), dim=-1)
        frame_logits = self.classifier(x)
        return torch.sum(att_weights * frame_logits, dim=-1), frame_logits

class BirdSED_Pretrained(nn.Module):
    def __init__(self, backbone_name, num_classes):
        super(BirdSED_Pretrained, self).__init__()
        
        self.audio_extractor = AudioToSpectrogramGPU()
        
        self.backbone = timm.create_model(
            model_name=backbone_name, 
            pretrained=False,
            in_chans=1, 
            num_classes=0,
            global_pool='' 
        )
        
        in_features = self.backbone.num_features
        
        self.freq_pool = nn.AdaptiveAvgPool2d((1, None))
        self.dropout = nn.Dropout(0.5)
        self.sed_head = AttentivePooling(in_channels=in_features, num_classes=num_classes)

    def forward(self, waveform):
        x = self.audio_extractor(waveform)
        x = self.backbone.forward_features(x)
        x = self.freq_pool(x).squeeze(2)
        x = self.dropout(x)
        clip_logits, _ = self.sed_head(x)
        return clip_logits

def get_padded_chunk(wf, start, end, chunk_samples):
    c_len = end - start
    pad_left, start_safe = max(0, -start), max(0, start)
    end_safe, pad_right = min(end, wf.shape[1]), max(0, end - wf.shape[1])
    chunk = wf[:, start_safe:end_safe]
    if pad_left > 0 or pad_right > 0: chunk = torch.nn.functional.pad(chunk, (pad_left, pad_right))
    return chunk

def apply_temporal_smoothing(probs_matrix, sigma=1.0):
    smoothed = np.zeros_like(probs_matrix)
    for class_idx in range(probs_matrix.shape[1]):
        smoothed[:, class_idx] = gaussian_filter1d(probs_matrix[:, class_idx], sigma=sigma)
    return smoothed

# Load EfficientNet-B2
model_effnet = BirdSED_Pretrained('tf_efficientnet_b2', num_classes=num_classes)
model_effnet.load_state_dict(torch.load(model_weights_effnet, map_location=device))
model_effnet.to(device).eval()

# Load ConvNeXt-Tiny
model_convnext = BirdSED_Pretrained('convnext_tiny', num_classes=num_classes)
model_convnext.load_state_dict(torch.load(model_weights_convnext, map_location=device))
model_convnext.to(device).eval()

sample_sub_path = os.path.join(base_path, 'sample_submission.csv')
sample_df = pd.read_csv(sample_sub_path)
expected_species_columns = sample_df.columns[1:] 

test_audio_dir = os.path.join(base_path, 'test_soundscapes')
predictions_dict = {}
species_to_idx = {sp: idx for idx, sp in enumerate(label_encoder.inverse_transform(range(num_classes)))}

if os.path.exists(test_audio_dir) and len([f for f in os.listdir(test_audio_dir) if f.endswith('.ogg')]) > 0:
    test_files = [f for f in os.listdir(test_audio_dir) if f.endswith('.ogg')]
    print(f"Found {len(test_files)} hidden test files. Processing dynamically on CPU...")
    
    for file in test_files:
        file_path = os.path.join(test_audio_dir, file)
        file_stem = file.replace('.ogg', '')
        
        try:
            waveform, sr = torchaudio.load(file_path)
            if waveform.shape[0] > 1: waveform = torch.mean(waveform, dim=0, keepdim=True)
            
            chunk_samples, shift_samples = 5 * 32000, int(0.5 * 32000)
            num_chunks = waveform.shape[1] // chunk_samples 
            file_probs = []
            
            batch_size = 8 
            
            for b in range(0, num_chunks, batch_size):
                w_base, w_left, w_right = [], [], []
                
                for i in range(b, min(b + batch_size, num_chunks)):
                    start, end = i * chunk_samples, (i + 1) * chunk_samples
                    w_base.append(get_padded_chunk(waveform, start, end, chunk_samples))
                    w_left.append(get_padded_chunk(waveform, start - shift_samples, end - shift_samples, chunk_samples))
                    w_right.append(get_padded_chunk(waveform, start + shift_samples, end + shift_samples, chunk_samples))
                    
                batch_base = torch.stack(w_base).to(device)
                batch_left = torch.stack(w_left).to(device)
                batch_right = torch.stack(w_right).to(device)
                
                with torch.no_grad():
                    # 1. EfficientNet TTA Preds
                    e_tta = (torch.sigmoid(model_effnet(batch_base)) + 
                             torch.sigmoid(model_effnet(batch_left)) + 
                             torch.sigmoid(model_effnet(batch_right))) / 3.0
                    
                    # 2. ConvNeXt TTA Preds
                    c_tta = (torch.sigmoid(model_convnext(batch_base)) + 
                             torch.sigmoid(model_convnext(batch_left)) + 
                             torch.sigmoid(model_convnext(batch_right))) / 3.0
                    
                    e_safe = np.clip(e_tta.cpu().numpy(), 1e-6, 1.0)
                    c_safe = np.clip(c_tta.cpu().numpy(), 1e-6, 1.0)
                    
                    probs_ensemble = (e_safe ** WEIGHT_EFFNET) * (c_safe ** WEIGHT_CONVNEXT)
                                     
                    file_probs.append(probs_ensemble)
            
            if len(file_probs) > 0:
                file_probs = np.vstack(file_probs)
                smoothed_probs = apply_temporal_smoothing(file_probs, sigma=1.0)
                
                for i in range(num_chunks):
                    predictions_dict[f"{file_stem}_{(i + 1) * 5}"] = smoothed_probs[i] 
                    
            del waveform
            gc.collect()
            
        except Exception as e:
            print(f"Error processing {file}: {e}")

for idx, row in sample_df.iterrows():
    r_id = row['row_id']
    if r_id in predictions_dict:
        pred_probs = predictions_dict[r_id]
        for col in expected_species_columns:
            if col in species_to_idx:
                sample_df.at[idx, col] = pred_probs[species_to_idx[col]]

sample_df.to_csv('submission.csv', index=False)
print("Submission.csv generated")